# 📚 Tarea Semana 5: Reconocimiento de Voz con Redes Neuronales Recurrentes (RNN)

## 🎯 Objetivo General
Desarrollar un sistema de **clasificación de audio** capaz de **identificar y categorizar diferentes tipos de voces** (incluyendo la propia voz del estudiante y la de al menos otras dos personas), así como el **silencio**, en un entorno de trabajo.

---

## ✅ Lo que debes hacer

1. **Diseñar una red neuronal con arquitectura RNN** (puedes usar LSTM o GRU) utilizando **Keras y TensorFlow**.
2. **Recolectar muestras de audio** de tu voz, la de dos personas más y también grabaciones de silencio.
3. **Preprocesar los datos** para que estén listos para el entrenamiento (normalización, extracción de características como MFCC, etc.).
4. **Entrenar el modelo** con tus datos usando la arquitectura RNN.
5. **Evaluar el modelo** con métricas como precisión y matriz de confusión.
6. **Presentar los resultados** en un notebook bien documentado.

---

## 📌 Objetivos Específicos

### 🎙️ 1. Recolectar y Preprocesar Datos de Audio
- Captura muestras de voz y silencio.
- Asegúrate de que los audios tengan buena calidad.
- Extrae características relevantes (por ejemplo, MFCC).

### 🧠 2. Diseñar y Entrenar el Modelo
- Implementa una red neuronal recurrente (RNN) usando **Keras** y **TensorFlow**.
- Puedes guiarte por el tutorial visto en clase y el material complementario.
- Entrena el modelo para clasificar correctamente las voces y el silencio.

### 🧪 3. Validar y Evaluar el Modelo
- Evalúa el rendimiento del modelo con métricas como **precisión**, **F1 Score** y **matriz de confusión**.
- Asegúrate de que el modelo sea **robusto y equilibrado** en sus predicciones.

---

## 🛠️ Recomendaciones Técnicas

- Usa **TensorFlow 2.x** y **Keras** para construir el modelo.
- Utiliza capas como `tf.keras.layers.LSTM` o `tf.keras.layers.GRU`.
- Para la clasificación final, usa `Dense` con activación `softmax`.
- Puedes usar herramientas como **Edge Impulse** si deseas experimentar con despliegue en dispositivos.

---

## 📁 Entregables

- Notebook en Jupyter o Google Colab con:
  - Código bien comentado.
  - Explicaciones claras de cada paso.
  - Resultados de evaluación.
  - Gráficas (matriz de confusión, precisión, etc.).
- Breve informe (opcional) explicando el proceso y los hallazgos.

---



In [ ]:
import os
import librosa
import soundfile as sf
import numpy as np

# Rutas
input_dir = 'dataset_voces'
output_dir = 'dataset_voces_procesado'

os.makedirs(output_dir, exist_ok=True)

target_sr = 16000
duration_sec = 1
samples_per_chunk = target_sr * duration_sec

for speaker_name in os.listdir(input_dir):
    speaker_dir = os.path.join(input_dir, speaker_name)
    if os.path.isdir(speaker_dir):
        print(f'Procesando locutor: {speaker_name}')
        out_speaker_dir = os.path.join(output_dir, speaker_name)
        os.makedirs(out_speaker_dir, exist_ok=True)
        chunk_index = 0
        for filename in os.listdir(speaker_dir):
            if filename.endswith('.wav'):
                file_path = os.path.join(speaker_dir, filename)
                try:
                    audio, sr = librosa.load(file_path, sr=target_sr, mono=True)
                    total_samples = len(audio)
                    for start in range(0, total_samples, samples_per_chunk):
                        end = start + samples_per_chunk
                        chunk = audio[start:end]
                        if len(chunk) < samples_per_chunk:
                            chunk = np.pad(chunk, (0, samples_per_chunk - len(chunk)), 'constant')
                        sf.write(os.path.join(out_speaker_dir, f'{chunk_index}.wav'), chunk, target_sr)
                        chunk_index += 1
                    print(f'  - {filename} dividido en {chunk_index} fragmentos.')
                except Exception as e:
                    print(f'  - Error con {filename}: {e}')


### 1. Visualización de Características de Audio

Antes de entrenar el modelo, es importante entender cómo la red neuronal "ve" el audio. Para ello utilizamos tres gráficas fundamentales:

1. **Forma de Onda (Waveform):** Es la representación más básica del sonido. Muestra cómo varía la amplitud (el volumen o la fuerza del sonido) a lo largo del tiempo. Nos ayuda a identificar silencios y momentos de mayor intensidad.
2. **Espectrograma:** Muestra las frecuencias presentes en el audio a lo largo del tiempo, y qué tan intensas son (representado por colores). Nos permite ver si un sonido es agudo o grave en un instante determinado.
3. **Coeficientes Cepstrales en las Frecuencias de Mel (MFCCs):** Es la representación que realmente usaremos para entrenar. Los MFCCs extraen las características más importantes de la voz humana, simulando cómo el oído humano percibe el sonido, eliminando la información irrelevante.

In [ ]:
import librosa.display
import matplotlib.pyplot as plt
import os

def plot_audio_features(audio_path):
    y, sr = librosa.load(audio_path, sr=None)
    speaker_name = os.path.basename(os.path.dirname(audio_path))
    
    plt.figure(figsize=(15, 10))
    
    # 1) Forma de onda
    plt.subplot(3, 1, 1)
    librosa.display.waveshow(y, sr=sr)
    plt.title(f'Forma de onda - {speaker_name}')
    
    # 2) Espectrograma
    plt.subplot(3, 1, 2)
    import numpy as np
    S = librosa.stft(y)
    D = librosa.amplitude_to_db(np.abs(S), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'Espectrograma - {speaker_name}')
    
    # 3) MFCCs
    plt.subplot(3, 1, 3)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    librosa.display.specshow(mfccs, x_axis='time')
    plt.colorbar()
    plt.title(f'MFCCs - {speaker_name}')
    
    plt.tight_layout()
    plt.show()

# Visualizamos un ejemplo de cada clase
ejemplos = [
    'dataset_voces_procesado/Gilmer/0.wav',
    'dataset_voces_procesado/Mario/0.wav',
    'dataset_voces_procesado/Oscar/0.wav',
    'dataset_voces_procesado/Silencio/0.wav'
]

for ruta in ejemplos:
    if os.path.exists(ruta):
        plot_audio_features(ruta)

### 2. Extracción de Características y Preparación de Datos
Aquí convertimos todos nuestros audios en representaciones numéricas (MFCCs) y los dividimos en datos de entrenamiento, validación y prueba.

In [ ]:
import tensorflow as tf   
from sklearn.model_selection import train_test_split  
from sklearn.preprocessing import LabelEncoder        
from sklearn.preprocessing import StandardScaler      
import numpy as np

parent_dir = "dataset_voces_procesado"
speaker_folders = ["Gilmer", "Mario", "Oscar", "Silencio"]

def extract_features(parent_dir, speaker_folders):
    features = []   
    labels = []     
    
    for i, speaker_folder in enumerate(speaker_folders):
        speaker_folder_path = os.path.join(parent_dir, speaker_folder)  
        for filename in os.listdir(speaker_folder_path):
            if filename.endswith(".wav"):  
                file_path = os.path.join(speaker_folder_path, filename)  
                
                audio, sr = librosa.load(file_path, sr=None, duration=1)
                mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
                mfccs = StandardScaler().fit_transform(mfccs)
                
                features.append(mfccs.T)
                labels.append(i)
                
    return np.array(features), np.array(labels)

X, y = extract_features(parent_dir, speaker_folders)

label_encoder = LabelEncoder()                       
y = label_encoder.fit_transform(y)                   
label_encoder.classes_ = np.array(speaker_folders)   

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)  
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)  

print("Forma de los datos de entrenamiento:", X_train.shape)
print("Forma de los datos de validación:", X_val.shape)

### 3. Definición y Entrenamiento del Modelo
Utilizamos una Red Neuronal Recurrente (RNN) con capa LSTM, ideal para datos secuenciales como el audio.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

model = tf.keras.Sequential([
    tf.keras.layers.LSTM(128, input_shape=(X_train.shape[1], X_train.shape[2])),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(len(speaker_folders), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=20, batch_size=32, callbacks=[early_stopping])

if early_stopping.stopped_epoch > 0:
    print("⏹️ Entrenamiento detenido tempranamente en la época", early_stopping.stopped_epoch + 1)
else:
    print("✅ Entrenamiento completado sin detención temprana")

plt.plot(history.history['loss'], label='Pérdida de entrenamiento')
plt.plot(history.history['val_loss'], label='Pérdida de validación')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.show()

### 4. Evaluación del Modelo
Calculamos la precisión del modelo en los datos de prueba y visualizamos una Matriz de Confusión.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, f1_score

y_pred_probabilities = model.predict(X_test)  
y_pred = np.argmax(y_pred_probabilities, axis=1)  

y_test_decoded = label_encoder.inverse_transform(y_test)
y_pred_decoded = label_encoder.inverse_transform(y_pred)

conf_matrix = confusion_matrix(y_test_decoded, y_pred_decoded, labels=speaker_folders)

accuracy = accuracy_score(y_test_decoded, y_pred_decoded)
print(f"Precisión en la evaluación del conjunto de prueba: {accuracy}")

f1 = f1_score(y_test_decoded, y_pred_decoded, labels=speaker_folders, average='weighted')
print(f"Puntaje F1 ponderado: {f1}")

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues",
            xticklabels=speaker_folders, yticklabels=speaker_folders)
plt.xticks(rotation=45, ha="right")
plt.title("Matriz de Confusión")
plt.xlabel("Etiqueta Predicha")
plt.ylabel("Etiqueta Verdadera")
plt.show()